In [1]:
# ============================================================
# ENTITYQUESTIONS RETRIEVAL RANK REPORT (RESEARCH FORMAT)
# ============================================================

import re
import json
import unicodedata
import numpy as np
from tqdm.auto import tqdm
from datasets import load_dataset

# -------------------------
# SETTINGS
# -------------------------
DATASET_NAME = "MinaGabriel/entityquestions-final-clean"
OUT_FILE = "SRE_entityquestions_retrieval_rank_report_tiers.txt"

TOP_K = 20
K_LIST = [1, 5, 10, 20]

LT_THRESHOLD = 100
FREQ_THRESHOLD = 10000

TEXT_KEY = "text"
ANS_KEY = "possible_answers"

# -------------------------
# NORMALIZATION
# -------------------------
_PUNCT = re.compile(r"[^\w\s]")
_WS = re.compile(r"\s+")

def norm(s):
    if s is None:
        return ""
    s = unicodedata.normalize("NFKD", str(s))
    s = s.lower()
    s = _PUNCT.sub(" ", s)
    s = _WS.sub(" ", s)
    return s.strip()

# -------------------------
# PARSE ANSWERS
# -------------------------
def parse_list(x):

    if x is None:
        return []

    if isinstance(x, list):
        return [str(a) for a in x]

    if isinstance(x, str):

        s = x.strip()

        if s.startswith("[") and s.endswith("]"):
            try:
                v = json.loads(s)
                if isinstance(v, list):
                    return [str(a) for a in v]
            except Exception:
                pass

        return [s]

    return [str(x)]

# -------------------------
# HIT RANK
# -------------------------
def first_hit_rank_top20(retrieved_docs, answers):

    answers = parse_list(answers)
    answers = [norm(a) for a in answers if len(norm(a)) >= 3]

    if not retrieved_docs or not answers:
        return None

    for i, d in enumerate(retrieved_docs[:TOP_K], start=1):

        text = d.get(TEXT_KEY, "") if isinstance(d, dict) else str(d)
        text = norm(text)

        if not text:
            continue

        for a in answers:

            if re.search(rf"\b{re.escape(a)}\b", text):
                return i

    return None

# -------------------------
# MEAN
# -------------------------
def mean01(xs):
    return float(np.mean(xs)) if xs else 0.0


# ============================================================
# LOAD DATASET
# ============================================================

ds = load_dataset(DATASET_NAME, split="train")

n = len(ds)

print("Dataset loaded:", n)


# ============================================================
# METRICS STORAGE
# ============================================================

metrics = {
    "ALL":       {"hits": {k: [] for k in K_LIST}, "n": 0},
    "LONG-TAIL": {"hits": {k: [] for k in K_LIST}, "n": 0},
    "INFREQUENT":{"hits": {k: [] for k in K_LIST}, "n": 0},
    "FREQUENT":  {"hits": {k: [] for k in K_LIST}, "n": 0},
}


# ============================================================
# MAIN LOOP
# ============================================================

for ex in tqdm(ds, total=n, desc="Evaluating R@k by Tiers"):

    gold = ex.get(ANS_KEY)
    retrieved = ex.get("retrieved_docs", [])

    s_pop = int(ex.get("s_pop", 0))

    hit_rank = first_hit_rank_top20(retrieved, gold)

    # Tier classification
    if s_pop < LT_THRESHOLD:
        tier = "LONG-TAIL"
    elif s_pop < FREQ_THRESHOLD:
        tier = "INFREQUENT"
    else:
        tier = "FREQUENT"

    for group in ["ALL", tier]:

        metrics[group]["n"] += 1

        for k in K_LIST:

            val = 1 if (hit_rank is not None and hit_rank <= k) else 0
            metrics[group]["hits"][k].append(val)


# ============================================================
# REPORT GENERATION
# ============================================================

lines = []

lines.append("=" * 80)
lines.append("EntityQuestions Retrieval Report — Research Tiers (Long-Tail, Infrequent, Frequent)")
lines.append("=" * 80)

lines.append(f"Split: train | n={n}")
lines.append(f"Method: stored retrieved_docs (top_k={TOP_K})")
lines.append("")

lines.append("Tier Definitions (by s_pop):")
lines.append(f"- LONG-TAIL:  s_pop < {LT_THRESHOLD}")
lines.append(f"- INFREQUENT: {LT_THRESHOLD} <= s_pop < {FREQ_THRESHOLD}")
lines.append(f"- FREQUENT:   s_pop >= {FREQ_THRESHOLD}")
lines.append("")


for name in ["ALL", "LONG-TAIL", "INFREQUENT", "FREQUENT"]:

    m = metrics[name]

    lines.append(f"{name:<12} (n={m['n']})")

    for k in K_LIST:

        score = mean01(m["hits"][k])
        lines.append(f"  R@{k:<2}  = {score:.4f}")

    lines.append("")


with open(OUT_FILE, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))


print("Saved tier-based report:", OUT_FILE)

Dataset loaded: 22075


Evaluating R@k by Tiers:   0%|          | 0/22075 [00:00<?, ?it/s]

Saved tier-based report: SRE_entityquestions_retrieval_rank_report_tiers.txt
